# Voice & Multimodal AI

Companion notebook for the [Voice & Multimodal lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/13-voice-and-multimodal-ai).

**The idea in one sentence.** Voice and multimodal systems add two challenges: a **latency
budget** (STT → LLM → TTS must feel real-time, so tail latency matters), and a **shared
embedding space** (CLIP-style) that lets you retrieve *images* from *text* and vice versa.

The building blocks:

- **Voice latency budget:** the pipeline latency is the *sum* of stage latencies, and the
  **p95 tail** is what users feel — one slow stage blows the budget.
- **Cross-modal retrieval:** CLIP maps images and text into one space, so a text query
  retrieves the matching images by cosine similarity.

We simulate both and **validate the latency budget and cross-modal retrieval**, then cover
the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(7)

## 1 — Voice AI latency budget

A voice AI pipeline chains VAD → STT → LLM → TTS. We model the latency distribution for each stage and compute end-to-end p95.

In [ ]:
# Simulate latency (ms) for each stage based on reported production numbers
n_requests = 10_000

# STT (streaming Whisper): mean 180ms, std 40ms
stt_lat  = rng.normal(180, 40, n_requests).clip(80, 500)
# LLM inference (GPT-4o): mean 280ms, std 80ms
llm_lat  = rng.normal(280, 80, n_requests).clip(100, 800)
# TTS first-chunk (streaming): mean 120ms, std 30ms
tts_lat  = rng.normal(120, 30, n_requests).clip(60, 300)
# Total pipeline latency
total    = stt_lat + llm_lat + tts_lat

for stage, lat in [("STT", stt_lat), ("LLM", llm_lat), ("TTS", tts_lat), ("Total", total)]:
    print(f"{stage:8s}: p50={np.percentile(lat,50):.0f}ms  p95={np.percentile(lat,95):.0f}ms  p99={np.percentile(lat,99):.0f}ms")

print(f"\nFraction of requests under 1 second: {(total < 1000).mean():.1%}")

plt.figure(figsize=(8, 3))
plt.hist(total, bins=60, color='#6366f1', alpha=0.8)
plt.axvline(np.percentile(total, 95), color='#f59e0b', lw=2, label=f'p95={np.percentile(total,95):.0f}ms')
plt.axvline(1000, color='#f43f5e', lw=2, ls='--', label='1s budget')
plt.xlabel('End-to-end latency (ms)'); plt.ylabel('Count')
plt.title('Voice AI Pipeline Latency Distribution')
plt.legend(); plt.tight_layout(); plt.show()

### Validate: pipeline latency is the sum of stages, and the tail dominates

The end-to-end latency is STT + LLM + TTS per request, so the pipeline p95 is driven by
the stages' tails — and is *worse* than any single stage's p95. We confirm the total is the
sum and that the p95 exceeds the mean (the tail users actually feel).

In [ ]:
assert np.allclose(total, stt_lat + llm_lat + tts_lat), 'total latency is the sum of the stages'
p50, p95 = np.percentile(total, 50), np.percentile(total, 95)
print(f'pipeline latency  mean {total.mean():.0f} ms, p50 {p50:.0f} ms, p95 {p95:.0f} ms')
print(f'stage means: STT {stt_lat.mean():.0f}, LLM {llm_lat.mean():.0f}, TTS {tts_lat.mean():.0f} ms')
assert p95 > p50, 'the p95 tail exceeds the median — what users actually feel'
assert total.mean() > max(stt_lat.mean(), llm_lat.mean(), tts_lat.mean()), 'the pipeline is slower than any one stage'
print('\n✅ latency adds across stages; the p95 tail (not the mean) is the real-time budget')

## 2 — Visual search with CLIP embeddings

CLIP encodes images and text into the same embedding space. Visual search is ANN retrieval over pre-encoded catalog image embeddings.

In [ ]:
# Simulate a catalog of 1000 "image embeddings" (CLIP d=512 proxy)
n_catalog, d = 1000, 64   # use d=64 for speed (CLIP is 512)
catalog_emb = rng.normal(size=(n_catalog, d))
catalog_emb /= np.linalg.norm(catalog_emb, axis=1, keepdims=True)

# Assign categories (fashion, home, electronics, food)
categories = ['fashion','home','electronics','food']
item_cats = np.array([categories[i % 4] for i in range(n_catalog)])

# "Query image" - embed it with the same CLIP encoder
# Simulate: a fashion item query
fashion_centroid = catalog_emb[item_cats == 'fashion'].mean(0)
fashion_centroid /= np.linalg.norm(fashion_centroid)
query_noise = rng.normal(0, 0.2, d)
query_emb = fashion_centroid + query_noise
query_emb /= np.linalg.norm(query_emb)

# Cosine similarity search
sims = catalog_emb @ query_emb
top_k = 10
top_ids = np.argsort(-sims)[:top_k]

print(f"Top-{top_k} visual search results:")
cat_counts = {c: 0 for c in categories}
for i in top_ids:
    cat_counts[item_cats[i]] += 1
    print(f"  Item {i:4d} [{item_cats[i]:12s}] sim={sims[i]:.4f}")

print(f"\nCategory distribution in top-{top_k}:")
for cat, cnt in cat_counts.items():
    print(f"  {cat}: {cnt}/{top_k} results")

### Validate: cosine retrieval ranks by similarity and finds an exact match

Visual search returns catalog items in descending cosine order, and querying with an
item's *own* embedding retrieves that exact item at rank 1 (recall@1 = 1). We verify both
— the core correctness of embedding retrieval. (Real CLIP embeddings also cluster by
*category*; this toy's catalog is random vectors, so we demonstrate the category-clustering
effect on a structured set in the next cell.)

In [ ]:
# results are ranked by descending cosine similarity
assert np.all(np.diff(sims[top_ids]) <= 1e-9), 'visual search returns results in descending similarity'
# querying with a catalog item's own embedding retrieves it at rank 1
probe = 137
q = catalog_emb[probe]
rank1 = int(np.argmax(catalog_emb @ q))
print(f'query = item {probe}\'s embedding -> top result is item {rank1} (cos = {(catalog_emb @ q).max():.3f})')
assert rank1 == probe, 'an exact-embedding query must retrieve itself at rank 1'
print('\n✅ cosine retrieval ranks by similarity and returns the exact match at rank 1')

## 3 — Cross-modal retrieval: text → image

CLIP's joint embedding space allows querying images with text (and vice versa). We simulate this with a category-aware "text encoder".

In [ ]:
def text_query_embedding(query_text, category_centroids):
    """
    Simulate a text encoder that maps text to the image embedding space.
    In real CLIP, this is done by the text transformer.
    """
    # Simple proxy: map category keywords to centroids
    for cat, centroid in category_centroids.items():
        if cat in query_text.lower():
            noise = rng.normal(0, 0.15, len(centroid))
            emb = centroid + noise
            return emb / np.linalg.norm(emb)
    return rng.normal(size=list(category_centroids.values())[0].shape)

# Pre-compute category centroids
centroids = {cat: catalog_emb[item_cats == cat].mean(0) for cat in categories}
for cat in centroids:
    centroids[cat] /= np.linalg.norm(centroids[cat])

text_queries = ["show me fashion items", "find home decor", "electronics on sale"]
for query in text_queries:
    q_emb = text_query_embedding(query, centroids)
    sims = catalog_emb @ q_emb
    top3 = np.argsort(-sims)[:3]
    cats = [item_cats[i] for i in top3]
    print(f"Query: '{query}' → top cats: {cats}")

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **tail latency** | one slow stage's p95 blows the real-time budget (verified) — stream and cap each stage |
| **error propagation** | STT mistakes feed the LLM garbage; the pipeline is only as good as its first stage |
| **modality gap** | text and image embeddings occupy different cones; retrieval is relative, not absolute |
| **turn-taking / barge-in** | voice UX needs interruption handling, not just low latency |
| **embedding dim mismatch** | both modalities must map to the *same* space to compare |

Demo: a text query retrieves images of the matching category — cross-modal CLIP retrieval.

In [ ]:
# What real CLIP embeddings do (that the random toy above can't): cluster by category so
# a TEXT query retrieves same-category IMAGES. We build a GENUINELY clustered catalog —
# per-category centroids + small noise — and show cross-modal text->image retrieval working.
rng_c = np.random.default_rng(0)
cat_centroids = rng_c.normal(size=(4, d)); cat_centroids /= np.linalg.norm(cat_centroids, axis=1, keepdims=True)
labels = np.repeat(np.arange(4), 250)
clustered = cat_centroids[labels] + 0.25 * rng_c.normal(size=(1000, d))
clustered /= np.linalg.norm(clustered, axis=1, keepdims=True)
# a 'text encoder' maps the word 'electronics' (category 2) into the same space
text_vec = cat_centroids[2] + 0.1 * rng_c.normal(size=d); text_vec /= np.linalg.norm(text_vec)
top = np.argsort(clustered @ text_vec)[::-1][:10]
frac = (labels[top] == 2).mean()
print(f'text query for category 2 -> {frac:.0%} of top-10 images are category 2')
assert frac > 0.8, 'in a properly clustered space, a text query retrieves same-category images'
print('\nWith real (clustered) embeddings, text and images share one space -> text retrieves images. That IS CLIP.')

## ✏️ Your turn

**Exercise.** Implement `cosine_recall_at_k(query_emb, catalog_emb, relevant_ids, k)`:
Given a query embedding and catalog, return the fraction of `relevant_ids` that appear in the top-k ANN results (Recall@K).

This is the primary metric for visual and semantic search quality.

In [ ]:
def cosine_recall_at_k(query_emb, catalog_emb, relevant_ids, k=10):
    """
    query_emb:   (d,) query embedding
    catalog_emb: (N, d) catalog embeddings
    relevant_ids: list of relevant item indices (ground truth)
    k: number of results to retrieve
    Returns: recall = |top_k ∩ relevant| / |relevant|
    """
    # TODO(you): compute cosine similarities, retrieve top-k, compute recall
    return ...

# Test: query is close to the first 20 fashion items
relevant = list(np.where(item_cats == 'fashion')[0][:20])
recall = cosine_recall_at_k(query_emb, catalog_emb, relevant, k=20)
print(f"Recall@20 for fashion query: {recall:.4f}")
print(f"Expected: > 0.3 (query is fashion-like)")

In [ ]:
# Assertion
assert 0.0 <= cosine_recall_at_k(query_emb, catalog_emb, relevant, k=20) <= 1.0
# Perfect case: query exactly equals one of the relevant items
perfect_query = catalog_emb[relevant[0]]
r = cosine_recall_at_k(perfect_query, catalog_emb, [relevant[0]], k=1)
assert abs(r - 1.0) < 1e-6, "Exact query should have Recall@1 = 1.0"
print("✓ cosine_recall_at_k correct")

<details><summary>Solution</summary>

```python
def cosine_recall_at_k(query_emb, catalog_emb, relevant_ids, k=10):
    sims = catalog_emb @ query_emb
    top_k_ids = set(np.argsort(-sims)[:k].tolist())
    hits = len(top_k_ids & set(relevant_ids))
    return hits / len(relevant_ids) if relevant_ids else 0.0
```
</details>

## Key takeaways

- **Voice latency adds across stages** (STT + LLM + TTS); the **p95 tail**, not the mean,
  is the real-time budget (verified) — one slow stage blows it.
- **A shared embedding space (CLIP) unifies modalities:** cosine retrieval ranks by
  similarity and finds exact matches (verified), and with genuinely clustered embeddings a
  text query retrieves same-category images (demo) — that's cross-modal retrieval.
- **Cross-modal retrieval works** because text and images land in the *same* space.
- **Streaming everywhere:** stream STT, LLM, and TTS to cut perceived latency.